# Phân tích lỗi tác tử — Issue #12

Thống kê và biểu đồ lỗi từ các run phân tích.

In [ ]:
import os
from pathlib import Path

# Tự tìm thư mục gốc project
root = Path.cwd()
while not (root / 'pyproject.toml').exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print('Root:', os.getcwd())

In [ ]:
import json
import statistics
from collections import defaultdict

import matplotlib.pyplot as plt
import pandas as pd

FIGS = Path('docs/figures')
FIGS.mkdir(parents=True, exist_ok=True)

# Nạp tất cả lỗi từ các run
ERROR_DIRS = [
    'runs/tournament/main_mode_a_l1_l6/error_analysis/errors.jsonl',
    'runs/tournament/main_mode_a_l7_additional/error_analysis/errors.jsonl',
    'runs/tournament/main_mode_b_l1_l6/error_analysis/errors.jsonl',
    'runs/tournament/main_mode_b_l7_additional/error_analysis/errors.jsonl',
    'runs/tournament/main_mode_a_l8_additional/error_analysis/errors.jsonl',
]

all_errors = []
for path in ERROR_DIRS:
    p = Path(path)
    if p.exists():
        errors = [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]
        all_errors.extend(errors)
        print(f'Đọc {len(errors)} lỗi từ {path}')
    else:
        print(f'Không tìm thấy: {path}')

df = pd.DataFrame(all_errors)
print(f'\nTổng: {len(df)} lỗi')

## Bảng thống kê lỗi theo loại

In [ ]:
by_type = df['error_type'].value_counts().reset_index()
by_type.columns = ['Loại lỗi', 'Số lỗi']
by_type['%'] = (by_type['Số lỗi'] / len(df) * 100).round(1)
print('### Bảng phân loại lỗi')
print(by_type.to_string(index=False))

## Bảng thống kê lỗi theo cấp

In [ ]:
by_agent = df['agent'].value_counts().reset_index()
by_agent.columns = ['Cấp', 'Số lỗi']
by_agent['%'] = (by_agent['Số lỗi'] / len(df) * 100).round(1)
print('### Bảng lỗi theo cấp')
print(by_agent.to_string(index=False))

## Biểu đồ lỗi theo loại

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Phân loại lỗi
type_counts = df['error_type'].value_counts()
axes[0].bar(type_counts.index, type_counts.values, color=['tomato','steelblue','seagreen','orange'])
axes[0].set_title('Phân loại lỗi tác tử')
axes[0].set_xlabel('Loại lỗi')
axes[0].set_ylabel('Số lỗi')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(type_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center')

# Biểu đồ 2: Lỗi theo cấp
agent_counts = df['agent'].value_counts()
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c','#e67e22']
axes[1].bar(agent_counts.index, agent_counts.values, color=colors[:len(agent_counts)])
axes[1].set_title('Số lỗi theo cấp tác tử')
axes[1].set_xlabel('Cấp')
axes[1].set_ylabel('Số lỗi')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(agent_counts.values):
    axes[1].text(i, v + 0.5, str(v), ha='center')

plt.tight_layout()
plt.savefig(FIGS / 'error_analysis_overview.png', dpi=150)
plt.show()
print(f'Lưu: {FIGS}/error_analysis_overview.png')

## Biểu đồ lỗi theo loại và cấp (heatmap)

In [ ]:
pivot = df.groupby(['agent', 'error_type']).size().unstack(fill_value=0)
print('### Bảng lỗi theo cấp × loại')
print(pivot.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, colormap='Set2')
ax.set_title('Phân loại lỗi theo cấp tác tử')
ax.set_xlabel('Cấp')
ax.set_ylabel('Số lỗi')
ax.tick_params(axis='x', rotation=15)
ax.legend(title='Loại lỗi', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig(FIGS / 'error_by_agent_type.png', dpi=150)
plt.show()
print(f'Lưu: {FIGS}/error_by_agent_type.png')

## Ván minh họa

In [ ]:
# Ván có nhiều lỗi nhất
game_error_counts = df.groupby('game_id').size().sort_values(ascending=False)
top_games = game_error_counts.head(5)
print('### Top 5 ván có nhiều lỗi nhất')
for game_id, count in top_games.items():
    game_errors = df[df['game_id'] == game_id]
    agent_x = game_errors['agent_x'].iloc[0]
    agent_o = game_errors['agent_o'].iloc[0]
    result  = game_errors['result'].iloc[0]
    winner  = 'X' if result == 1 else 'O' if result == -1 else 'Hòa'
    print(f'  game_id={game_id}: {agent_x} vs {agent_o}, thắng={winner}, {count} lỗi')
    for _, e in game_errors.iterrows():
        print(f'    ply={e["ply"]} {e["agent"]} {e["error_type"]} move={e["move"]} best={e["best_move"]} drop={e["score_drop"]}')

## Kết luận phân tích lỗi

In [ ]:
total = len(df)
top_error = df['error_type'].value_counts().index[0]
top_agent = df['agent'].value_counts().index[0]
least_errors = df['agent'].value_counts().index[-1]

print(f"""
=== KẾT LUẬN PHÂN TÍCH LỖI ===

Tổng số lỗi phát hiện: {total} lỗi
Dữ liệu: main_mode_a (L1-L8), main_mode_b (L1-L7)
Phương pháp: AlphaBeta depth=2 đánh giá lại từng nước

Loại lỗi phổ biến nhất: {top_error}
Cấp mắc nhiều lỗi nhất: {top_agent}
Cấp ít lỗi nhất: {least_errors}

Nhận xét:
- L1-Random: không phân tích (ngẫu nhiên hoàn toàn)
- L2-Rule: lỗi chiến lược cao do chỉ nhìn 1 nước
- L4-MCTS: bỏ sót thắng ngay do giới hạn mô phỏng
- L5/L6: lỗi phức tạp hơn, khó phát hiện bằng depth=2
- L7/L8: ít lỗi nhất nhưng vẫn bỏ sót thắng ngay

Hạn chế:
- Đánh giá lại bằng AlphaBeta depth=2 không hoàn hảo với L5-L8
- Chưa phát hiện được lỗi VCF và ba-ba kép phức tạp
- Chưa có mode_b L8 để so sánh đầy đủ
""")